In [2]:
!pip -q install transformers accelerate pillow pandas scikit-learn tqdm sentencepiece

In [3]:
import os, re, json
from pathlib import Path
import pandas as pd
import torch
from tqdm import tqdm
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Paths (FIXED from Experiment 1)
BASE_DIR = Path('/content/drive/MyDrive/colab_experiments')
DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load utilities by direct execution (bypass import issues)
print("Loading utilities...")

with open(BASE_DIR / 'utils/data_loader.py') as f:
    exec(f.read(), globals())

with open(BASE_DIR / 'utils/prompt_templates.py') as f:
    exec(f.read(), globals())

with open(BASE_DIR / 'utils/metrics.py') as f:
    exec(f.read(), globals())

print("✓ Utilities loaded successfully")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading utilities...
✓ Utilities loaded successfully


In [5]:
# Load metadata with explanations
metadata_df = load_metadata(DATA_DIR / 'metadata.csv', DATA_DIR / 'explanations.csv')

print(f'✓ Loaded {len(metadata_df)} rows')
print(f'✓ Columns: {list(metadata_df.columns)}')

# Check for missing explanations
if 'explanation_implicit' in metadata_df.columns:
    missing = metadata_df['explanation_implicit'].isna().sum()
    print(f'✓ Missing explanations: {missing}/{len(metadata_df)}')

    # Show sample
    print('\n=== Sample Row ===')
    sample = metadata_df.iloc[0]
    print(f"Image: {sample['image_id']}")
    print(f"Explanation: {sample['explanation_implicit'][:200]}...")
    print(f"Label: {sample['human_label']}")
else:
    print('⚠️ No explanation_implicit column found!')

metadata_df.head(2)

✓ Loaded 650 rows
✓ Columns: ['image_id', 'category', 'ocr_text', 'human_label', 'image_path', 'explanation_implicit', 'human_label_vec']
✓ Missing explanations: 69/650

=== Sample Row ===
Image: TE-104.jpg
Explanation: CONTEXT: This is a bilingual (Hindi/English) Indian meme using a simple cartoon template to express the feeling that even comforting activities, like eating favorite foods, cannot fix underlying emoti...
Label: [1,0,0,0,0,0,0]


,image_id,category,ocr_text,human_label,image_path,explanation_implicit,human_label_vec
0,TE-104.jpg,BRAND_DEPENDENT,इतनी tastyBiryani खाने के बादभी HEAL MY FEELINGS,"[1,0,0,0,0,0,0]",test_images/TE-104.jpg,CONTEXT: This is a bilingual (Hindi/English) I...,"[1, 0, 0, 0, 0, 0, 0]"
1,TE-177.jpg,BRAND_DEPENDENT,Porn hub 0 Previous 102 ^ 103| 101!* /05N 105|...,"[1,0,0,0,1,0,0]",test_images/TE-177.jpg,1. MOOD/EMOTIONAL STATE: The text uses a mocki...,"[1, 0, 0, 0, 1, 0, 0]"


In [6]:
# Reload metadata with corrected explanations
metadata_df = load_metadata(DATA_DIR / 'metadata.csv', DATA_DIR / 'explanations.csv')

print(f'✓ Loaded {len(metadata_df)} rows')
print(f'✓ Columns: {list(metadata_df.columns)}')

# Check for missing explanations
if 'explanation_implicit' in metadata_df.columns:
    has_explanation = (metadata_df['explanation_implicit'] != '') & (metadata_df['explanation_implicit'].notna())
    with_exp = has_explanation.sum()
    without_exp = len(metadata_df) - with_exp

    print(f'✓ With explanations: {with_exp}/650 ({with_exp/650*100:.1f}%)')
    print(f'✓ Without explanations: {without_exp}/650 ({without_exp/650*100:.1f}%)')

    # Show sample with explanation
    print('\n=== SAMPLE ROW WITH EXPLANATION ===')
    sample_with = metadata_df[has_explanation].iloc[0]
    print(f"Image: {sample_with['image_id']}")
    print(f"Explanation: {sample_with['explanation_implicit'][:250]}...")
    print(f"Label: {sample_with['human_label']}")

    if with_exp >= 580:
        print('\n✅ EXCELLENT! 89%+ coverage - ready for PALO-7B inference!')
    else:
        print(f'\n⚠️ Only {with_exp/650*100:.1f}% coverage')

metadata_df.head(2)

✓ Loaded 650 rows
✓ Columns: ['image_id', 'category', 'ocr_text', 'human_label', 'image_path', 'explanation_implicit', 'human_label_vec']
✓ With explanations: 581/650 (89.4%)
✓ Without explanations: 69/650 (10.6%)

=== SAMPLE ROW WITH EXPLANATION ===
Image: TE-104.jpg
Explanation: CONTEXT: This is a bilingual (Hindi/English) Indian meme using a simple cartoon template to express the feeling that even comforting activities, like eating favorite foods, cannot fix underlying emotional distress.

1. MOOD/EMOTIONAL STATE: The chara...
Label: [1,0,0,0,0,0,0]

✅ EXCELLENT! 89%+ coverage - ready for PALO-7B inference!


,image_id,category,ocr_text,human_label,image_path,explanation_implicit,human_label_vec
0,TE-104.jpg,BRAND_DEPENDENT,इतनी tastyBiryani खाने के बादभी HEAL MY FEELINGS,"[1,0,0,0,0,0,0]",test_images/TE-104.jpg,CONTEXT: This is a bilingual (Hindi/English) I...,"[1, 0, 0, 0, 0, 0, 0]"
1,TE-177.jpg,BRAND_DEPENDENT,Porn hub 0 Previous 102 ^ 103| 101!* /05N 105|...,"[1,0,0,0,1,0,0]",test_images/TE-177.jpg,1. MOOD/EMOTIONAL STATE: The text uses a mocki...,"[1, 0, 0, 0, 1, 0, 0]"


In [7]:
# Upgrade transformers to latest version
print("Upgrading transformers to support PALO-7B...")
!pip install -q --upgrade transformers accelerate

print("\n✓ Upgrade complete!")
print("Note: May need to restart runtime after this")

Upgrading transformers to support PALO-7B...

✓ Upgrade complete!
Note: May need to restart runtime after this


In [10]:
import sys
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download

model_id = 'MBZUAI/PALO-7B'

print("=== LOADING PALO-7B WITH CUSTOM CODE ===\n")

# Step 1: Download entire repo to access custom modeling files
print("Step 1: Downloading PALO-7B repository...")
model_path = snapshot_download(
    repo_id=model_id,
    cache_dir="/content/palo_cache",
    local_dir="/content/PALO-7B",
    local_dir_use_symlinks=False
)
print(f"✓ Downloaded to: {model_path}")

# Step 2: Add to path
sys.path.insert(0, model_path)
print("✓ Added to Python path")

# Step 3: Import custom model class
print("\nStep 2: Importing custom PALO model class...")
try:
    from modeling_palo import PaloForConditionalGeneration
    from configuration_palo import PaloConfig
    print("✓ Custom classes imported")

    # Step 4: Load model
    print("\nStep 3: Loading model...")
    processor = AutoTokenizer.from_pretrained(model_path)
    model = PaloForConditionalGeneration.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map='auto'
    )
    model.eval()

    print("\n✓ PALO-7B loaded successfully!")
    print(f"✓ Device: {next(model.parameters()).device}")

except ImportError as e:
    print(f"✗ Import failed: {e}")
    print("\nLet me check what files are available...")
    import os
    files = os.listdir(model_path)
    print(f"Files in repo: {[f for f in files if f.endswith('.py')]}")

=== LOADING PALO-7B WITH CUSTOM CODE ===

Step 1: Downloading PALO-7B repository...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

✓ Downloaded to: /content/PALO-7B
✓ Added to Python path

Step 2: Importing custom PALO model class...
✗ Import failed: No module named 'modeling_palo'

Let me check what files are available...
Files in repo: []


In [11]:
import os

print("=== CHECKING PALO-7B REPO CONTENTS ===\n")

repo_path = "/content/PALO-7B"
all_files = []
for root, dirs, files in os.walk(repo_path):
    for file in files:
        rel_path = os.path.relpath(os.path.join(root, file), repo_path)
        all_files.append(rel_path)

print(f"Total files: {len(all_files)}\n")

# Show important files
print("Key files:")
for f in sorted(all_files):
    if any(x in f for x in ['config.json', 'README', '.py', 'model', 'processor']):
        print(f"  {f}")

# Check config.json specifically
config_path = os.path.join(repo_path, "config.json")
if os.path.exists(config_path):
    import json
    with open(config_path) as f:
        config = json.load(f)
    print(f"\n=== config.json contents ===")
    print(json.dumps(config, indent=2))

=== CHECKING PALO-7B REPO CONTENTS ===

Total files: 21

Key files:
  .cache/huggingface/download/README.md.metadata
  .cache/huggingface/download/config.json.metadata
  .cache/huggingface/download/generation_config.json.metadata
  .cache/huggingface/download/pytorch_model-00001-of-00002.bin.metadata
  .cache/huggingface/download/pytorch_model-00002-of-00002.bin.metadata
  .cache/huggingface/download/pytorch_model.bin.index.json.metadata
  .cache/huggingface/download/tokenizer.model.metadata
  .cache/huggingface/download/tokenizer_config.json.metadata
  README.md
  config.json
  generation_config.json
  pytorch_model-00001-of-00002.bin
  pytorch_model-00002-of-00002.bin
  pytorch_model.bin.index.json
  tokenizer.model
  tokenizer_config.json

=== config.json contents ===
{
  "_name_or_path": "lmsys/vicuna-7b-v1.5",
  "architectures": [
    "PaloForCausalLM"
  ],
  "bos_token_id": 1,
  "eos_token_id": 2,
  "freeze_mm_mlp_adapter": false,
  "hidden_act": "silu",
  "hidden_size": 4096,
  

In [12]:
# Create summary of PALO-7B attempt
summary = """
# EXPERIMENT 2 ATTEMPT - PALO-7B (FAILED)

## Date: 2026-04-01

## Objective
Run explanation-augmented inference using MBZUAI/PALO-7B with 650 Hindi mental health memes.

## Model Configuration
- Model: MBZUAI/PALO-7B
- GPU: NVIDIA A100-SXM4-40GB (Google Colab Pro)
- Approach: Explanation + Image input (Track B)

## Explanations Prepared
- Source: phase1_explanations_v2.csv
- Total images: 650
- With explanations: 581/650 (89.4%)
- Without explanations: 69/650 (10.6%)

## Loading Attempts

### Attempt 1: Standard AutoModelForCausalLM
```python
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)
```
**Result:** FAILED
**Error:** `ValueError: model type 'palo' not recognized by transformers`

### Attempt 2: Upgrade Transformers + Reload
- Upgraded transformers to latest version
- Restarted runtime
- Re-attempted loading
**Result:** FAILED (same error)

### Attempt 3: Direct Repo Download
- Downloaded full PALO-7B repo using snapshot_download
- Attempted to import custom modeling classes
- Expected files: modeling_palo.py, configuration_palo.py
**Result:** FAILED
**Finding:** Custom .py files NOT included in HuggingFace repo

## Root Cause Analysis
1. PALO-7B requires custom modeling code (PaloForCausalLM class)
2. Custom code NOT distributed with model weights on HuggingFace
3. Model built with transformers v4.31.0 (2023) - outdated
4. Likely requires cloning from research GitHub repo with complex setup
5. Not designed for easy Colab deployment

## Model Architecture (from config.json)
- Base: lmsys/vicuna-7b-v1.5
- Custom class: PaloForCausalLM
- Vision tower: openai/clip-vit-large-patch14-336
- Hidden size: 4096
- Projection: mlp2x_gelu
- Model weights: 14GB (successfully downloaded)

## Time Invested
- Setup and troubleshooting: ~30 minutes
- Multiple loading attempts: 3
- Model download: 14GB (successful but unusable)

## Decision
**SKIP PALO-7B** - pursuing alternative model for Experiment 2

## Alternative Approach
Will use LLaVA-NeXT-7B or BLIP-2 for explanation-augmented inference instead.
Both models:
- Are well-supported by transformers library
- Can process explanation + image input
- Test same scientific hypothesis (Track B: explanation-augmented inference)
- Will produce comparable results

## Research Impact
- No impact on research validity
- Track B (explanation-augmented) can be tested with alternative model
- PALO-7B specifically is not critical to research questions
- Can mention attempt in paper as model selection consideration

## Lessons Learned
1. Research models on HuggingFace may require custom code not in repo
2. Check model compatibility with transformers before committing time
3. Have backup models identified for critical experiments
4. LLaVA and BLIP families are more production-ready

## Files Generated
- None (model loading failed before inference)

## Next Steps
- Proceed with Experiment 2 using alternative model
- Expected completion: Today
- Expected improvement over baseline: 35-45% macro-F1
"""

# Save summary
summary_path = RESULTS_DIR / 'palo7b_attempt_summary.txt'
with open(summary_path, 'w') as f:
    f.write(summary)

print(summary)
print(f"\n✓ Summary saved to: {summary_path}")

# Download it
from google.colab import files
files.download(str(summary_path))

print("\n✓ Download initiated - check your browser's download folder")


# EXPERIMENT 2 ATTEMPT - PALO-7B (FAILED)

## Date: 2026-04-01

## Objective
Run explanation-augmented inference using MBZUAI/PALO-7B with 650 Hindi mental health memes.

## Model Configuration
- Model: MBZUAI/PALO-7B
- GPU: NVIDIA A100-SXM4-40GB (Google Colab Pro)
- Approach: Explanation + Image input (Track B)

## Explanations Prepared
- Source: phase1_explanations_v2.csv
- Total images: 650
- With explanations: 581/650 (89.4%)
- Without explanations: 69/650 (10.6%)

## Loading Attempts

### Attempt 1: Standard AutoModelForCausalLM
```python
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)
```
**Result:** FAILED
**Error:** `ValueError: model type 'palo' not recognized by transformers`

### Attempt 2: Upgrade Transformers + Reload
- Upgraded transformers to latest version
- Restarted runtime
- Re-attempted loading
**Result:** FAILED (same error)

### Attempt 3: Direct Repo Downlo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Download initiated - check your browser's download folder
